# Preliminary Code for DS6050 Group 4 Project

## NOTE : Will likely be best to seperate these out into seperate .py files, this is simply for early confirmation. Every cell is already written to operate as own .py folder with exception of function imports/dataset download

In [10]:
# code block for import statements and packages used throughout
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms

import numpy as np
import os
import kagglehub
import copy

In [3]:
# data download cell, only need to run if you are downloading new data, could also make your own .env file but this is quicker
# just make sure to remove any actual values before saving/sharing.
os.environ['KAGGLE_USERNAME'] = "your_username"
os.environ['KAGGLE_KEY'] = "your_api_key"

path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Path to dataset files:", path)

Path to dataset files: /home/kay5hv/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2


## Block to define the model architecture as given in the document (seperate .py)

### NOTE : Is this the architecture we want to use? It's the one defined in the paper and seems fine but just confirming

In [13]:
#define the Residual Blocks
class resBlock(nn.Module):
    # adding kernel size to this if want to change it
    def __init__(self,in_channels,out_channels,kernel_size = 3,stride=1,padding=1):
        super(resBlock, self).__init__()
        # for this first conv need to determine exact parameters for padding/other metrics. 
        # No need for bias since we apply batch norm after.
        # applying the "stride" value here since it seems to say in the description we only
        # apply the stride 2 on the first conv of the second/third stages?
        self.first_conv = nn.Conv2d(in_channels,out_channels,kernel_size,padding=1,stride = stride, bias = False)
        # apply the batchnorm
        self.bn1 = nn.BatchNorm2d(out_channels)
        # apply activation function
        self.relu = nn.ReLU(inplace=True)
        self.second_conv = nn.Conv2d(out_channels,out_channels,kernel_size,padding=1,stride = 1, bias = False)
        # apply the second batchnorm
        self.bn2 = nn.BatchNorm2d(out_channels)
        #building the skip connection with a check for size mismatch
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self,x):
        # pass x through first conv layer
        holder = self.first_conv(x)
        # normalize with first batch norm
        holder = self.bn1(holder)
        # apply relu activation function
        holder = self.relu(holder)
        # apply second conv layer
        holder = self.second_conv(holder)
        # normalize with second batch norm
        holder = self.bn2(holder)
        #skip connection
        holder = holder + self.shortcut(x)
        #return the final value after relu
        return self.relu(holder)

# define the actual model architecture
class model(nn.Module):
    #initialize with default value of 4 classes consistent with dataset
    #adding in_channel as well incase of different image type, but should be 3 channel
    def __init__(self,num_classes=4, in_channel = 3):
        #defining the different types of blocks to be used throughout 
        super(model,self).__init__()
        self.start_block = nn.Sequential(
        nn.Conv2d(in_channel, 32, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True)
        )
        self.resBlock1 = resBlock(32,64)
        self.resBlock2 = resBlock(64,128,stride=2)
        self.resBlock3 = resBlock(128,128,stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        # pass x through the start block layer
        holder = self.start_block(x)
        # add first resBlock 
        holder = self.resBlock1(holder)
        # add second resBlock 
        holder = self.resBlock2(holder)
        # add third resBlock 
        holder = self.resBlock3(holder)
        # use GAP to reduce
        holder = self.avgpool(holder)
        # flatten so to get it down to the required dimensionality
        holder = torch.flatten(holder, 1)
        # return the result of the linear layer
        return self.fc(holder)

## This will be the body/runner.py/data loader

### NOTE : Need to check initialization values here for the dataloaders/transforms. 

In [15]:
# defining the batch size at 64 by default and image_size at (224,224) for resizing since there are multiple sizes throughout the dataset
# could be worth seeing other resizing regimes to reduce data loss
def load_brain_tumor_data(base_path: str, batch_size: int = 64, image_size: tuple = (224, 224)):
    """Loads training and testing image datasets from a structured directory. Make sure that the version 2 subfolders of training/testing are 
  pulled into pwd.

  Example:
  base_path/
  ├── training/
  │   ├── glioma/
  │   ├── meningioma/
  │   ├── notumor/
  │   └── pituitary/
  └── testing/
      ├── glioma/
      ├── meningioma/
      ├── notumor/
      └── pituitary/
    """
    train_dir = os.path.join(base_path, 'Training/')
    test_dir = os.path.join(base_path, 'Testing/')

    """
    Note : The following section has the training/testing transforms. The actual transforms that are listed in our paper can 
    be applied to the data augmentation line below with relevant ones or even extended.
    
    For the normalization, right now it is set at [0.5,0.5,0.5] just naively. We will need to compute the actual values for the
    combined dataset.
    """
    # Define transformations (Resize, convert to Tensor, and Normalize)
    train_transforms = transforms.Compose([
          transforms.Resize(image_size),
          transforms.RandomHorizontalFlip(),  # Data augmentation for training
          transforms.ToTensor(),
          transforms.Normalize(
              mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]
          ),
      ])
    
    test_transforms = transforms.Compose([
          transforms.Resize(image_size),
          transforms.ToTensor(),
          transforms.Normalize(
              mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]
          ),
      ])
    
    # Load train/test datasets using ImageFolder
    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
    test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)
    
    # Wrap in DataLoaders for batching and shuffling using cuda acceleration to load if available
    # Could be worth playing with the num_workers to try to speed it up since on Rivanna we have the memory
    train_loader = DataLoader(
          dataset=train_dataset,
          batch_size=batch_size,
          shuffle=True,
          num_workers=4,
          pin_memory=True if torch.cuda.is_available() else False,
      )
    
    test_loader = DataLoader(
          dataset=test_dataset,
          batch_size=batch_size,
          shuffle=False,
          num_workers=4,
          pin_memory=True if torch.cuda.is_available() else False,
      )
    
    return train_dataset, test_dataset, train_loader, test_loader

# defining a simple patience early stopper to prevent it running extra epochs with no improvement, could look into other methods here
class EarlyStopping:
    """Monitors validation loss and stops training after a set patience."""
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.best_model_weights = None

    def __call__(self, val_loss, model):
        # Check if validation loss improved by at least min_delta
        if val_loss < (self.best_loss - self.min_delta):
            self.best_loss = val_loss
            self.counter = 0
            # Deepcopy weights to preserve best checkpoint in memory
            self.best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), 'best_brain_tumor_model.pth')
            print(f"--> Validation loss improved to {val_loss:.4f}. Checkpoint saved.")
        else:
            self.counter += 1
            print(f"--> EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

# simple validation using the model on the validation set/batch
def validate(model, test_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            # Accumulate loss (normalized by batch size)
            running_loss += loss.item() * images.size(0)

            # Track prediction accuracy
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    # Calculate metrics
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100.0
    return epoch_loss, epoch_acc

# Actual Runner
if __name__ == '__main__':
    data_path = './'
    train_ds, test_ds, train_loader, test_loader = load_brain_tumor_data(data_path)
    # Adding a function to simply map the class to the folder we pulled the image from as the class
    idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
    # Returning the class mapping just for knowledge
    print('Class Mapping:', idx_to_class)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Using Device",device)
    # Initializing Model, Loss, Optimizer, and Early Stopping
    my_model = model(num_classes=4, in_channel=3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(my_model.parameters(), lr=1e-3)
    early_stopper = EarlyStopping(patience=3)

    # number of epochs, set to 30 right now but feel free to change
    num_epochs = 30

    # Training Loop
    for epoch in range(num_epochs):
        my_model.train()
        train_loss = 0.0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = my_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            train_total += labels.size(0)

        avg_train_loss = train_loss / train_total

        # quick validation check
        val_loss, val_acc = validate(my_model, test_loader, criterion, device)

        #Checking each Epoch to see results
        # might be nice to add like a better "progress" bar / better metrics to see if certain classes are failing or what
        print(f"\n[Epoch {epoch+1:02d}/{num_epochs:02d}] "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.2f}%")

        # Checking Early Stopping
        early_stopper(val_loss, my_model)

        if early_stopper.early_stop:
            print("\n[!] Early stopping triggered: 3 periods of non-improving validation loss.")
            # Restore the best model weights
            my_model.load_state_dict(early_stopper.best_model_weights)
            break

    print("\nTraining complete. Best model weights restored and saved to 'best_brain_tumor_model.pth'.")


Class Mapping: {0: 'glioma', 1: 'meningioma', 2: 'notumor', 3: 'pituitary'}
Using Device cuda

[Epoch 01/30] Train Loss: 0.7727 | Val Loss: 18.6610 | Val Acc: 24.89%
--> Validation loss improved to 18.6610. Checkpoint saved.

[Epoch 02/30] Train Loss: 0.6507 | Val Loss: 10.7185 | Val Acc: 24.89%
--> Validation loss improved to 10.7185. Checkpoint saved.

[Epoch 03/30] Train Loss: 0.6032 | Val Loss: 2.0944 | Val Acc: 47.42%
--> Validation loss improved to 2.0944. Checkpoint saved.

[Epoch 04/30] Train Loss: 0.5346 | Val Loss: 1.1406 | Val Acc: 58.56%
--> Validation loss improved to 1.1406. Checkpoint saved.

[Epoch 05/30] Train Loss: 0.5105 | Val Loss: 1.4790 | Val Acc: 57.13%
--> EarlyStopping counter: 1/3

[Epoch 06/30] Train Loss: 0.4635 | Val Loss: 1.2256 | Val Acc: 61.05%
--> EarlyStopping counter: 2/3

[Epoch 07/30] Train Loss: 0.4456 | Val Loss: 2.8251 | Val Acc: 48.41%
--> EarlyStopping counter: 3/3

[!] Early stopping triggered: 5 periods of non-improving validation loss.

Trai